In [5]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit_aer import StatevectorSimulator

from qlbm.circuits import encode, encode_links, lcu_success_probability
from qlbm.physics import get_collision_diagonal, bc_config_to_matrix, combine_collision_bc_matrices, collision_nonuniform
from qlbm.analysis import count_gates, transpile_stage
from qlbm.initial_conditions import *

simulator = StatevectorSimulator()
EAGLE_BASIS = ['rz', 'sx', 'x', 'cx']

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Config

In [6]:
# D1Q3
links_1d = [[-1], [0], [1]]
weights_1d = [1/6, 2/3, 1/6]
speed_of_sound = 1 / np.sqrt(3)

# D2Q5
links_2d = [[0, 0], [1, 0], [-1, 0], [0, 1], [0, -1]]
weights_2d = [1/3, 1/6, 1/6, 1/6, 1/6]

EAGLE_BASIS = ['rz', 'sx', 'x', 'cx']

# ── 1D configs ──────────────────────────────────────────────────────
configs_1d = {}
for N in [4, 8]:
    density = get_gaussian_initial_distribution(N, center=N // 2, width=1.5, amplitude=0.9, background=0.1)
    density[0] = 0.0; density[-1] = 0.0
    velocity = get_uniform_velocity_field(N, velocity=0.3)
    bc = create_bounceback_bc_1d(N, links_1d)
    configs_1d[f'D1Q3 N={N}'] = dict(initial_density=density, velocity=velocity, bc=bc,
                                      links=links_1d, weights=weights_1d)

# ── 2D config (4x4, top wall BC) ────────────────────────────────────
Nx2, Ny2 = 4, 4
density_2d = np.full((Nx2, Ny2), 0.05)
density_2d[Nx2 // 2, :] = 0.9
density_2d[:, -1] = 0.0

nv = len(links_2d)
bc_2d = np.zeros((Nx2, Ny2, nv, nv))
for x in range(Nx2):
    for y in range(Ny2):
        bc_2d[x, y] = np.eye(nv)

link_dirs = {tuple(l): i for i, l in enumerate(links_2d)}
STAT, RIGHT, LEFT, UP, DOWN = [link_dirs[k] for k in [(0,0),(1,0),(-1,0),(0,1),(0,-1)]]

for x in range(1, Nx2 - 1):  # top wall
    bc_2d[x, Ny2-2, UP, UP] = 0.0
    bc_2d[x, Ny2-2, DOWN, UP] = 0.2
    bc_2d[x, Ny2-2, LEFT, UP] = 0.4
    bc_2d[x, Ny2-2, RIGHT, UP] = 0.4

velocity_2d = np.zeros((2, Nx2, Ny2))
velocity_2d[1, :, :] = 0.3

configs_2d = {
    'D2Q5 4x4': dict(initial_density=density_2d, velocity=velocity_2d, bc=bc_2d,
                     links=links_2d, weights=weights_2d, grid_size=(Nx2, Ny2))
}

Velocity indices: left=0, stationary=1, right=2
Velocity indices: left=0, stationary=1, right=2


## Success probability

In [7]:
results = {}
all_configs = {**configs_1d, **configs_2d}

for name, cfg in all_configs.items():
    initial_density = cfg['initial_density']
    velocity = cfg['velocity']
    bc = cfg['bc']
    links = cfg['links']
    weights = cfg['weights']
    grid_size = cfg.get('grid_size', (len(initial_density),))

    num_links = len(links)
    site_qubits_per_dim = [int(np.ceil(np.log2(s))) for s in grid_size]
    site_qubits = sum(site_qubits_per_dim)
    link_qubits = int(np.ceil(np.log2(num_links)))
    grid_size_flat = int(np.prod(grid_size))

    collision_diag = get_collision_diagonal(link_qubits, links, weights, velocity, speed_of_sound)

    ndim_spatial = bc.ndim - 2
    if ndim_spatial > 1:
        perm = list(reversed(range(ndim_spatial))) + [ndim_spatial, ndim_spatial + 1]
        bc_flat = np.ascontiguousarray(np.transpose(bc, perm)).reshape(grid_size_flat, num_links, num_links)
    else:
        bc_flat = bc
    bc_matrix = bc_config_to_matrix(bc_flat, grid_size_flat, 2**link_qubits)
    combined = combine_collision_bc_matrices(collision_diag, bc_matrix)

    el_gate = encode_links(link_qubits, num_links).to_gate(label='encode_links')

    # ── Merged: encode → encode_links → collision(BC@coll) ──
    nq_m = site_qubits + link_qubits + 1
    qc_m = QuantumCircuit(nq_m)
    qc_m.initialize(encode(initial_density, link_qubits))

    qc_t = QuantumCircuit(nq_m)
    qc_t.append(el_gate, list(range(site_qubits, site_qubits + link_qubits)))
    qc_m.compose(transpile(qc_t, simulator), inplace=True)

    col_gate = collision_nonuniform(site_qubits, link_qubits, combined).to_gate(label='collision')
    qc_t = QuantumCircuit(nq_m)
    qc_t.append(col_gate, list(range(nq_m)))
    qc_m.compose(transpile(qc_t, simulator), inplace=True)

    sv_m = simulator.run(qc_m).result().get_statevector()
    p_merged = lcu_success_probability(sv_m, num_ancillas=1)

    # ── Separate: encode → encode_links → collision(diag) → BC_LCU ──
    nq_s = site_qubits + link_qubits + 2
    qc_s = QuantumCircuit(nq_s)
    qc_s.initialize(encode(initial_density, link_qubits, extra_ancillas=1))

    qc_t = QuantumCircuit(nq_s)
    qc_t.append(el_gate, list(range(site_qubits, site_qubits + link_qubits)))
    qc_s.compose(transpile(qc_t, simulator), inplace=True)

    anc_coll = site_qubits + link_qubits
    col_diag_gate = collision_nonuniform(site_qubits, link_qubits, collision_diag).to_gate(label='collision_diag')
    qc_t = QuantumCircuit(nq_s)
    qc_t.append(col_diag_gate, list(range(site_qubits + link_qubits)) + [anc_coll])
    qc_s.compose(transpile(qc_t, simulator), inplace=True)

    anc_bc = site_qubits + link_qubits + 1
    bc_lcu_gate = collision_nonuniform(site_qubits, link_qubits, bc_matrix).to_gate(label='bc_lcu')
    qc_t = QuantumCircuit(nq_s)
    qc_t.append(bc_lcu_gate, list(range(site_qubits + link_qubits)) + [anc_bc])
    qc_s.compose(transpile(qc_t, simulator), inplace=True)

    sv_s = simulator.run(qc_s).result().get_statevector()
    p_separate = lcu_success_probability(sv_s, num_ancillas=2)

    results[name] = {'p_merged': p_merged, 'p_separate': p_separate}
    print(f"{name}:  P(merged) = {p_merged:.6f}  P(separate) = {p_separate:.6f}")

Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
D1Q3 N=4:  P(merged) = 0.370025  P(separate) = 0.226756
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
D1Q3 N=8:  P(merged) = 0.265388  P(separate) = 0.162633
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
D2Q5 4x4:  P(merged) = 0.308965  P(separate) = 0

## Gate count comparison

In [8]:
all_rows = []
all_configs = {**configs_1d, **configs_2d}

for name, cfg in all_configs.items():
    initial_density = cfg['initial_density']
    velocity = cfg['velocity']
    bc = cfg['bc']
    links = cfg['links']
    weights = cfg['weights']
    grid_size = cfg.get('grid_size', (len(initial_density),))

    num_links = len(links)
    site_qubits_per_dim = [int(np.ceil(np.log2(s))) for s in grid_size]
    site_qubits = sum(site_qubits_per_dim)
    link_qubits = int(np.ceil(np.log2(num_links)))
    grid_size_flat = int(np.prod(grid_size))

    collision_diag = get_collision_diagonal(link_qubits, links, weights, velocity, speed_of_sound)

    ndim_spatial = bc.ndim - 2
    if ndim_spatial > 1:
        perm = list(reversed(range(ndim_spatial))) + [ndim_spatial, ndim_spatial + 1]
        bc_flat = np.ascontiguousarray(np.transpose(bc, perm)).reshape(grid_size_flat, num_links, num_links)
    else:
        bc_flat = bc
    bc_matrix = bc_config_to_matrix(bc_flat, grid_size_flat, 2**link_qubits)
    combined = combine_collision_bc_matrices(collision_diag, bc_matrix)

    for approach, stages_spec in [
        ("merged", [
            ("collision (BC@coll)", collision_nonuniform(site_qubits, link_qubits, combined),
             site_qubits + link_qubits + 1, list(range(site_qubits + link_qubits + 1))),
        ]),
        ("separate", [
            ("collision (diag)", collision_nonuniform(site_qubits, link_qubits, collision_diag),
             site_qubits + link_qubits + 2, list(range(site_qubits + link_qubits)) + [site_qubits + link_qubits]),
            ("bc_lcu", collision_nonuniform(site_qubits, link_qubits, bc_matrix),
             site_qubits + link_qubits + 2, list(range(site_qubits + link_qubits)) + [site_qubits + link_qubits + 1]),
        ]),
    ]:
        for label, circuit, nq, qubits in stages_spec:
            gate = circuit.to_gate(label=label)
            r_aer = transpile_stage(label, gate, nq, qubits)
            r_eagle = transpile_stage(label, gate, nq, qubits, basis_gates=EAGLE_BASIS)

            all_rows.append({
                'name': name, 'approach': approach, 'stage': label,
                'aer_gates': r_aer['total_gates'], 'aer_depth': r_aer['depth'],
                'eagle_gates': r_eagle['total_gates'], 'eagle_depth': r_eagle['depth'],
            })

df = pd.DataFrame(all_rows)

for name in all_configs:
    print(f"\n{'='*80}")
    print(f"{name}  |  P(merged)={results[name]['p_merged']:.4f}  P(separate)={results[name]['p_separate']:.4f}")
    print(f"{'='*80}")
    sub = df[df['name'] == name]
    display(sub[['approach', 'stage', 'aer_gates', 'aer_depth', 'eagle_gates', 'eagle_depth']])

    for approach in ['merged', 'separate']:
        a = sub[sub['approach'] == approach]
        print(f"  {approach:10s} — Aer: {a['aer_gates'].sum():>8,} gates, depth {a['aer_depth'].sum():>8,}"
              f"  |  Eagle: {a['eagle_gates'].sum():>8,} gates, depth {a['eagle_depth'].sum():>8,}")

Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit

D1Q3 N=4  |  P(merged)=0.3700  P(separate)=0.2268


,approach,stage,aer_gates,aer_depth,eagle_gates,eagle_depth
0,merged,collision (BC@coll),1,1,2282,1432
1,separate,collision (diag),1,1,395,273
2,separate,bc_lcu,1,1,2247,1412


  merged     — Aer:        1 gates, depth        1  |  Eagle:    2,282 gates, depth    1,432
  separate   — Aer:        2 gates, depth        2  |  Eagle:    2,642 gates, depth    1,685

D1Q3 N=8  |  P(merged)=0.2654  P(separate)=0.1626


,approach,stage,aer_gates,aer_depth,eagle_gates,eagle_depth
3,merged,collision (BC@coll),1,1,9451,6008
4,separate,collision (diag),1,1,788,543
5,separate,bc_lcu,1,1,9435,5996


  merged     — Aer:        1 gates, depth        1  |  Eagle:    9,451 gates, depth    6,008
  separate   — Aer:        2 gates, depth        2  |  Eagle:   10,223 gates, depth    6,539

D2Q5 4x4  |  P(merged)=0.3090  P(separate)=0.2272


,approach,stage,aer_gates,aer_depth,eagle_gates,eagle_depth
6,merged,collision (BC@coll),1,1,154585,99300
7,separate,collision (diag),1,1,3162,2140
8,separate,bc_lcu,1,1,154529,99244


  merged     — Aer:        1 gates, depth        1  |  Eagle:  154,585 gates, depth   99,300
  separate   — Aer:        2 gates, depth        2  |  Eagle:  157,691 gates, depth  101,384
